# Gemma 4 Legal — Serving Inference Eval + CPU Fallback
**Updated: April 9, 2026**

Full evaluation notebook covering all 4 serving backends + CPU fallback chain with KAG/RAG/DAG retrieval context injection, L1–L4 KV cache compression tiers, Intel 10th Gen iGPU detection, LiteRT-LM MTP inference, and TurboQuant GPU benchmarking.

---

## 3-Track Pipeline Reference
```
Track 1 (NVIDIA RTX 3060 Ti):  turboquant_plus CUDA build → llama-server :8090 → benchmark turbo3 vs Q8_0 KV
Track 2 (Intel 10th Gen CPU):  litert-lm E4B (3.65 GB) → :8070 sidecar → MTP 4-head speedup test
Track 3 (Colab A100):          Gemma4_E4B_Legal_VLM_Reattach.ipynb → merge GRPO adapter → export GGUF + LiteRT
```

## Inference Fallback Chain
```
Client Query
  ↓
L1 (LokiJS, 5-10 min)  → L2 (IndexedDB, 7-day)  → L3 (Redis)  → L4 (Qdrant/pgvector)
                                                                         ↓ hit
                                                        KAG / RAG / DAG context assembled
                                                                         ↓
Tier 1: Gemma 4 E2B 2.3B (Transformers.js + WebGPU)
Tier 2: LiteRT-LM E2B/E4B  (:8070, XNNPACK CPU + Intel iGPU OpenCL,  MTP 4 heads, 1.8x speedup)
Tier 3: ONNX Gemma 3 270M  (WASM SIMD, legacy fallback)
Tier 4: TurboQuant llama-server (:8090, turbo3 KV, RTX 3060 Ti)
Tier 5: Ollama gemma4-legal  (:11434, VLM via mmproj GGUF)
```

## KV Cache L1–L4 Compression (Intel 10th gen reference)
| Level | Format | KV bits | Context @ i5-10500 L3 (12MB) | Context @ 8GB VRAM |
|-------|--------|---------|-------------------------------|---------------------|
| L1    | fp16   | 16 bit  | ~32 tokens                    | ~2048 tokens        |
| L2    | Q8_0   | 8 bit   | ~64 tokens                    | ~4096 tokens        |
| L3    | turbo3 | 3 bit   | ~256 tokens                   | ~24576 tokens       |
| L4    | turbo4 | 2 bit   | ~384 tokens                   | ~32768 tokens       |

## §§ Sections in This Notebook
- **§0** Environment Setup + Timestamp Logging
- **§1** L1–L4 Compression Configuration
- **§2** KAG / RAG / DAG Pipeline Setup
- **§3** Intel 10th Gen CPU Detection + Fallback Routing
- **§4** LiteRT-LM Health Check + MTP Inference
- **§5** TurboQuant Server Routing (GPU path)
- **§6** 5-Tier Inference Router with Fallback Logic
- **§7** Benchmark: turbo3 vs Q8_0 KV (CPU vs GPU)
- **§8** GRPO Adapter Merge + GGUF Export
- **§9** Inference Logging + Observability

## §0 — Environment Setup + Timestamp Logging
**Like setting up your workbench before the experiment — install tools, start the clock, keep a session diary.**

Install `litert-lm` for CPU inference, `llama-cpp-python` for GGUF loading, `faiss-cpu` for RAG retrieval.  
A `log_event()` helper keeps a running diary of every step with timestamps so you can replay what happened.

In [ ]:
import subprocess, sys

def pip_install(*packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

# Core inference + retrieval dependencies
pip_install(
    'litert-lm',           # Google LiteRT-LM (XNNPACK CPU + OpenCL iGPU)
    'llama-cpp-python',    # GGUF loading + llama-server client
    'transformers>=4.51',  # Gemma 4 base weights
    'peft',                # GRPO LoRA adapter merging
    'faiss-cpu',           # RAG dense retrieval
    'requests',            # HTTP sidecar calls (:8070, :8090)
    'psutil',              # RAM / CPU detection
    'cpuinfo',             # CPU model + generation detection
    'matplotlib',          # Benchmark charts
    'pandas',              # Results DataFrame
    'torch',               # Merge + export
    'accelerate',          # Multi-device model loading
)

print('✓ Dependencies installed')

# ── Timestamp + session log ───────────────────────────────────────────────
from datetime import datetime, timezone

SESSION_START = datetime.now(timezone.utc)
SESSION_LOG: list[dict] = []

def log_event(tag: str, msg: str) -> None:
    entry = {
        'ts': datetime.now(timezone.utc).isoformat(),
        'tag': tag,
        'msg': msg,
    }
    SESSION_LOG.append(entry)
    print(f"[{entry['ts']}] [{tag}] {msg}")

log_event('INIT', f'Session started — {SESSION_START.strftime("%A %B %d, %Y %H:%M UTC")}')
log_event('TRACKS', 'Track 1=TurboQuant:8090  Track 2=LiteRT:8070  Track 3=OllamaVLM:11434')

## §1 — L1–L4 Compression Configuration
**Like choosing what size container to pack lunch in — bigger container fits more but weighs more.**

- **L1 (fp16)**: full precision, smallest context, fastest decode per token
- **L2 (Q8_0)**: 8-bit KV, 2× context vs L1, good for Intel 10th gen RAM budget
- **L3 (turbo3)**: 3-bit KV via random rotation + codebook, 5× compression, fits 256 tokens in 12MB L3 cache
- **L4 (turbo4)**: 2-bit KV, maximum compression, slight quality loss at very long contexts

`pick_compression_level()` selects automatically based on available RAM + VRAM.

In [ ]:
from enum import IntEnum
from dataclasses import dataclass

class CompressionLevel(IntEnum):
    L1 = 1  # fp16 — full precision KV
    L2 = 2  # Q8_0 — 8-bit KV (baseline llama.cpp default)
    L3 = 3  # turbo3 — 3-bit KV via TurboQuant (ICLR 2026)
    L4 = 4  # turbo4 — 2-bit KV (maximum compression)

@dataclass
class CompressionProfile:
    level: CompressionLevel
    kv_bits: int
    kv_format: str                # llama-server -ctk / -ctv flag
    ctx_l3_12mb: int              # tokens that fit in Intel 10th gen 12MB L3 cache
    ctx_vram_8gb: int             # tokens that fit in 8GB VRAM
    ctx_ram_16gb: int             # tokens that fit in 16GB DDR4 RAM fallback
    note: str

COMPRESSION_PROFILES: dict[CompressionLevel, CompressionProfile] = {
    CompressionLevel.L1: CompressionProfile(
        level=CompressionLevel.L1, kv_bits=16, kv_format='f16',
        ctx_l3_12mb=32,    ctx_vram_8gb=2048,   ctx_ram_16gb=8192,
        note='Full precision — highest quality, smallest context on constrained hardware',
    ),
    CompressionLevel.L2: CompressionProfile(
        level=CompressionLevel.L2, kv_bits=8, kv_format='q8_0',
        ctx_l3_12mb=64,    ctx_vram_8gb=4096,   ctx_ram_16gb=16384,
        note='Q8_0 baseline — default llama.cpp, good RAM efficiency on Intel 10th gen',
    ),
    CompressionLevel.L3: CompressionProfile(
        level=CompressionLevel.L3, kv_bits=3, kv_format='turbo3',
        ctx_l3_12mb=256,   ctx_vram_8gb=24576,  ctx_ram_16gb=49152,
        note='TurboQuant turbo3 — 99.5% attention fidelity, 5× compression, L3-cache friendly',
    ),
    CompressionLevel.L4: CompressionProfile(
        level=CompressionLevel.L4, kv_bits=2, kv_format='turbo4',
        ctx_l3_12mb=384,   ctx_vram_8gb=32768,  ctx_ram_16gb=65536,
        note='TurboQuant turbo4 — maximum compression, small quality loss at very long contexts',
    ),
}

def pick_compression_level(available_ram_gb: float, vram_gb: float = 0.0) -> CompressionProfile:
    """Auto-select compression level based on available hardware."""
    if vram_gb >= 8:
        level = CompressionLevel.L3   # RTX 3060 Ti → turbo3 for 24K context
    elif available_ram_gb >= 16:
        level = CompressionLevel.L2   # 16GB DDR4 → Q8_0 comfortable
    elif available_ram_gb >= 8:
        level = CompressionLevel.L2   # 8GB min → Q8_0, limited context
    else:
        level = CompressionLevel.L1   # <8GB → fp16, max 2K context
    profile = COMPRESSION_PROFILES[level]
    log_event('COMPRESSION', f'Selected {profile.level.name} ({profile.kv_format}, {profile.kv_bits}-bit KV) — RAM={available_ram_gb}GB VRAM={vram_gb}GB')
    return profile

# Preview
for p in COMPRESSION_PROFILES.values():
    print(f'{p.level.name} ({p.kv_format:7s}): L3-cache ctx={p.ctx_l3_12mb:4d}t  VRAM-8GB ctx={p.ctx_vram_8gb:6d}t  RAM ctx={p.ctx_ram_16gb:6d}t')

## §2 — KAG / RAG / DAG Pipeline Setup
**Like three different research assistants: RAG checks the filing cabinet, KAG cross-references the encyclopedia, DAG maps out what depends on what first.**

- **RAG**: dense vector retrieval via FAISS — finds the 5 most similar case chunks
- **KAG**: entity-linked knowledge augmentation — expands query with statute + citation entities
- **DAG**: dependency-ordered multi-hop — resolves what needs answering first before the main question

The unified `retrieve(query, strategy)` dispatcher picks the right assistant.

In [ ]:
import numpy as np
from typing import Literal

# ── Lightweight in-process vector store (FAISS stub) ─────────────────────
# In production, this queries Qdrant on :6333 via the server API
# Here we use a simple FAISS flat index over sample embeddings

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    log_event('WARN', 'faiss-cpu not installed — RAG will use dummy retrieval')

DIMS = 768  # embeddinggemma:latest output dimensions

# Dummy corpus for notebook demo (replace with real Qdrant queries)
DEMO_CORPUS = [
    'FRE 901 requires authentication of evidence before admission.',
    'Chain of custody must be unbroken from collection to court presentation.',
    'Brady v. Maryland mandates disclosure of exculpatory evidence to defense.',
    'Hearsay is generally inadmissible unless an exception applies under FRE 803.',
    'The exclusionary rule bars evidence obtained in violation of constitutional rights.',
    'Fourth Amendment protects against unreasonable searches without a valid warrant.',
    'Stare decisis requires courts to follow binding precedent from higher courts.',
    'Federal Rule of Evidence 902 covers self-authenticating documents.',
]

if FAISS_AVAILABLE:
    _index = faiss.IndexFlatIP(DIMS)
    _corpus_vecs = np.random.randn(len(DEMO_CORPUS), DIMS).astype('float32')
    faiss.normalize_L2(_corpus_vecs)
    _index.add(_corpus_vecs)

# ── Shared prompt builder ─────────────────────────────────────────────────

def _build_rag_context(retrieved: list[str]) -> str:
    return '\n\n'.join(f'[{i+1}] {c}' for i, c in enumerate(retrieved))

# ── RAG: dense retrieval ─────────────────────────────────────────────────

def retrieve_rag(query: str, k: int = 3) -> str:
    """Dense vector retrieval from FAISS (→ Qdrant in production)."""
    if not FAISS_AVAILABLE:
        return f'[RAG-stub] No FAISS index. Query was: {query}'
    q_vec = np.random.randn(1, DIMS).astype('float32')
    faiss.normalize_L2(q_vec)
    _, idxs = _index.search(q_vec, k)
    chunks = [DEMO_CORPUS[i] for i in idxs[0] if i < len(DEMO_CORPUS)]
    log_event('RAG', f'Retrieved {len(chunks)} chunks for: {query[:60]}...')
    return _build_rag_context(chunks)

# ── KAG: knowledge-augmented entity linking ─────────────────────────────

_LEGAL_ENTITIES = {
    'hearsay':         'FRE 801-807 (Hearsay Rules)',
    'chain of custody': 'Evidence authentication — FRE 901-902',
    'brady':           'Brady v. Maryland 373 U.S. 83 (1963)',
    'fourth amendment': 'U.S. Const. amend. IV — Search and Seizure',
    'stare decisis':   'Common law doctrine of precedent',
    'exclusionary rule': 'Mapp v. Ohio 367 U.S. 643 (1961)',
}

def retrieve_kag(query: str, k: int = 3) -> str:
    """Entity-linked knowledge augmentation."""
    q_lower = query.lower()
    matched = [(kw, ref) for kw, ref in _LEGAL_ENTITIES.items() if kw in q_lower][:k]
    rag_ctx = retrieve_rag(query, k=2)
    entity_ctx = '\n'.join(f'[Entity] {kw} → {ref}' for kw, ref in matched) if matched else '[Entity] No direct entity match'
    log_event('KAG', f'Linked {len(matched)} entities for: {query[:60]}...')
    return f'{entity_ctx}\n\n{rag_ctx}'

# ── DAG: dependency-ordered multi-hop resolution ──────────────────────────

def retrieve_dag(query: str) -> str:
    """Dependency-resolved multi-hop retrieval."""
    # Phase 1: identify sub-questions
    sub_q = [
        f'What is the legal definition relevant to: {query}',
        f'What precedent applies to: {query}',
    ]
    # Phase 2: retrieve per sub-question, combine
    parts = [retrieve_rag(sq, k=2) for sq in sub_q]
    log_event('DAG', f'Resolved {len(sub_q)} sub-questions for: {query[:60]}...')
    return '\n\n--- Sub-question 2 ---\n\n'.join(parts)

# ── Unified dispatcher ───────────────────────────────────────────────────

def retrieve(query: str, strategy: Literal['rag', 'kag', 'dag'] = 'rag') -> str:
    if strategy == 'rag':
        return retrieve_rag(query)
    elif strategy == 'kag':
        return retrieve_kag(query)
    elif strategy == 'dag':
        return retrieve_dag(query)
    raise ValueError(f'Unknown strategy: {strategy}')

# Test
sample = 'Is hearsay evidence admissible under the federal rules?'
print('=== RAG ===')
print(retrieve(sample, 'rag'))
print('\n=== KAG ===')
print(retrieve(sample, 'kag')[:300])

## §3 — Intel 10th Gen CPU Detection + Fallback Routing
**Like checking what kind of kitchen you have before deciding which recipe to cook — gas stove (NVIDIA), electric (Intel iGPU), or camping stove (CPU only).**

`is_intel_10th_gen()` reads the CPU model string and checks for Comet Lake (10th gen) signature.  
`get_intel_igpu_device()` uses OpenVINO to probe for Intel UHD 620/630 iGPU availability.  
These helpers drive the fallback router in §6 to pick the right inference path per machine.

In [ ]:
import psutil
import platform
from dataclasses import dataclass, field

try:
    import cpuinfo
    _cpu_info = cpuinfo.get_cpu_info()
    CPU_BRAND = _cpu_info.get('brand_raw', platform.processor())
except Exception:
    CPU_BRAND = platform.processor()

@dataclass
class HardwareProfile:
    cpu_brand: str
    ram_gb: float
    is_intel_10th: bool
    has_igpu: bool
    igpu_device: str
    vram_gb: float
    compression: 'CompressionProfile' = field(default=None)

def is_intel_10th_gen() -> bool:
    """Detect Intel 10th gen (Comet Lake) by CPU brand string."""
    markers = ['-10', 'i3-10', 'i5-10', 'i7-10', 'i9-10', 'comet lake', '10th gen']
    brand = CPU_BRAND.lower()
    matched = any(m in brand for m in markers)
    return matched

def get_intel_igpu_device() -> tuple[bool, str]:
    """Probe for Intel iGPU via OpenVINO runtime."""
    try:
        from openvino.runtime import Core  # type: ignore
        core = Core()
        devices = core.available_devices
        igpu_devices = [d for d in devices if 'GPU' in d.upper()]
        if igpu_devices:
            return True, igpu_devices[0]
    except Exception:
        pass
    return False, 'CPU'

def get_nvidia_vram_gb() -> float:
    """Estimate NVIDIA VRAM via nvidia-smi subprocess."""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=5,
        )
        if result.returncode == 0:
            return int(result.stdout.strip().split('\n')[0]) / 1024
    except Exception:
        pass
    return 0.0

def detect_hardware() -> HardwareProfile:
    ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    intel_10th = is_intel_10th_gen()
    has_igpu, igpu_dev = get_intel_igpu_device()
    vram_gb = get_nvidia_vram_gb()
    profile = HardwareProfile(
        cpu_brand=CPU_BRAND,
        ram_gb=round(ram_gb, 1),
        is_intel_10th=intel_10th,
        has_igpu=has_igpu,
        igpu_device=igpu_dev,
        vram_gb=round(vram_gb, 1),
    )
    profile.compression = pick_compression_level(ram_gb, vram_gb)
    log_event('HW', f'CPU={CPU_BRAND!r}  10th-gen={intel_10th}  iGPU={igpu_dev}  RAM={ram_gb:.1f}GB  VRAM={vram_gb:.1f}GB')
    return profile

HW = detect_hardware()
print(f'CPU         : {HW.cpu_brand}')
print(f'Intel 10th  : {HW.is_intel_10th}')
print(f'iGPU        : {HW.igpu_device} (available={HW.has_igpu})')
print(f'RAM         : {HW.ram_gb} GB')
print(f'VRAM        : {HW.vram_gb} GB')
print(f'Compression : {HW.compression.level.name} ({HW.compression.kv_format})')

## §4 — LiteRT-LM Health Check + MTP Inference
**Like calling the nearby bakery to check they're open before driving there — and when you get there, they serve you 4 pastries at once (MTP speculative heads) instead of one at a time.**

LiteRT-LM runs as an HTTP sidecar on `:8070`. The `is_litert_ready()` health check caches the answer for 60 seconds.  
MTP (Multi-Token Prediction): 4 speculative draft heads predict the next 4 tokens simultaneously → ~1.8× speedup.  
Works on Intel UHD 620/630 via OpenCL backend, or falls back to XNNPACK SIMD on x86 CPU.

In [ ]:
import requests
import time
from functools import lru_cache

LITERT_BASE_URL = 'http://127.0.0.1:8070'
LITERT_MTP_HEADS = 4
LITERT_MTP_SPEEDUP = 1.8  # expected 1.8× vs sequential decode

_litert_ready: bool | None = None
_litert_check_expiry: float = 0.0

def is_litert_ready(timeout: float = 3.0) -> bool:
    """Ping :8070/health with 60-second TTL cache."""
    global _litert_ready, _litert_check_expiry
    if _litert_ready is not None and time.time() < _litert_check_expiry:
        return _litert_ready
    try:
        r = requests.get(f'{LITERT_BASE_URL}/health', timeout=timeout)
        _litert_ready = r.ok
    except Exception:
        _litert_ready = False
    _litert_check_expiry = time.time() + 60
    log_event('LITERT', f'Health check → {"UP" if _litert_ready else "DOWN (not running)"}')
    return _litert_ready

def litert_complete(prompt: str, context: str = '', max_tokens: int = 256) -> dict:
    """
    Call LiteRT-LM sidecar with pre-assembled KAG/RAG/DAG context.
    Returns dict with 'text', 'tokens', 'latency_ms'.
    """
    if not is_litert_ready():
        return {'text': None, 'tokens': 0, 'latency_ms': 0, 'error': 'LiteRT sidecar not running'}

    full_prompt = f'{context}\n\n{prompt}' if context else prompt
    payload = {
        'prompt': full_prompt,
        'max_tokens': max_tokens,
        'temperature': 0.1,
        'mtp_draft_heads': LITERT_MTP_HEADS,   # 4 speculative heads
    }
    t0 = time.perf_counter()
    try:
        r = requests.post(f'{LITERT_BASE_URL}/v1/completions', json=payload, timeout=120)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        if not r.ok:
            return {'text': None, 'tokens': 0, 'latency_ms': elapsed_ms, 'error': r.text}
        data = r.json()
        text = data.get('choices', [{}])[0].get('text', '')
        tokens = data.get('usage', {}).get('completion_tokens', len(text.split()))
        tok_per_sec = tokens / (elapsed_ms / 1000) if elapsed_ms > 0 else 0
        log_event('LITERT', f'{tokens} tokens in {elapsed_ms:.0f}ms → {tok_per_sec:.1f} tok/s  (MTP={LITERT_MTP_HEADS} heads)')
        return {'text': text, 'tokens': tokens, 'latency_ms': elapsed_ms, 'tok_per_sec': tok_per_sec}
    except Exception as e:
        return {'text': None, 'tokens': 0, 'latency_ms': 0, 'error': str(e)}

# ── Test: LiteRT + KAG context injected ──────────────────────────────────
SAMPLE_QUERY = 'What is the chain of custody requirement for DNA evidence?'
kag_ctx = retrieve(SAMPLE_QUERY, strategy='kag')

if is_litert_ready():
    result = litert_complete(SAMPLE_QUERY, context=kag_ctx)
    print(f'LiteRT response ({result["tok_per_sec"]:.1f} tok/s):')
    print(result['text'])
else:
    print('⚠ LiteRT-LM sidecar not running.')
    print('Run: litert-lm serve ./litert-models/gemma4-e2b/gemma-4-E2B-it.litertlm --port 8070 --backend=cpu')
    print(f'\nKAG context that would be injected:\n{kag_ctx[:400]}...')

## §5 — TurboQuant Server Routing (GPU path — Track 1)
**Like a super-efficient postal sorting machine that uses 3-bit postcodes instead of full zip codes — still gets to the right house, uses 5× less space.**

TurboQuant: ICLR 2026 training-free KV cache compression. 3-bit values via random rotation + codebook.  
`llama-server` fork at `:8090` with `-ctk turbo3 -ctv turbo3` flags.  
SSE streaming via `data:` lines (standard OpenAI-compatible format).

**Track 1 setup:**
```bash
llama-server -m gemma4-legal-vlm-Q4_K_M.gguf -ctk turbo3 -ctv turbo3 --port 8090 --n-gpu-layers 35
```

In [ ]:
import json as _json
from typing import Iterator

TURBOQUANT_BASE_URL = 'http://127.0.0.1:8090'

TurboQuantLevel = Literal['turbo2', 'turbo3', 'turbo4']

TURBOQUANT_CONTEXT_LIMITS = {
    'turbo2': {'vram_8gb': 16384, 'cpu_l3_12mb': 128, 'kv_bits': 4},
    'turbo3': {'vram_8gb': 24576, 'cpu_l3_12mb': 256, 'kv_bits': 3},
    'turbo4': {'vram_8gb': 32768, 'cpu_l3_12mb': 384, 'kv_bits': 2},
}

_turboquant_ready: bool | None = None
_turboquant_check_expiry: float = 0.0

def is_turboquant_ready(timeout: float = 3.0) -> bool:
    global _turboquant_ready, _turboquant_check_expiry
    if _turboquant_ready is not None and time.time() < _turboquant_check_expiry:
        return _turboquant_ready
    try:
        r = requests.get(f'{TURBOQUANT_BASE_URL}/health', timeout=timeout)
        _turboquant_ready = r.ok
    except Exception:
        _turboquant_ready = False
    _turboquant_check_expiry = time.time() + 60
    log_event('TURBO', f'TurboQuant :8090 → {"UP" if _turboquant_ready else "DOWN"}')
    return _turboquant_ready

def try_turboquant(
    prompt: str,
    context: str = '',
    level: TurboQuantLevel = 'turbo3',
    max_tokens: int = 256,
    stream: bool = False,
) -> dict:
    """POST to TurboQuant llama-server (:8090, OpenAI-compatible v1/completions)."""
    if not is_turboquant_ready():
        return {'text': None, 'tokens': 0, 'latency_ms': 0, 'error': 'TurboQuant server not running'}

    full_prompt = f'{context}\n\n{prompt}' if context else prompt
    ctx_limit = TURBOQUANT_CONTEXT_LIMITS[level]['vram_8gb']
    payload = {
        'model': 'gemma4-legal',
        'prompt': full_prompt,
        'max_tokens': max_tokens,
        'temperature': 0.1,
        'stream': stream,
    }
    t0 = time.perf_counter()
    try:
        r = requests.post(f'{TURBOQUANT_BASE_URL}/v1/completions', json=payload,
                          stream=stream, timeout=120)
        elapsed_ms = (time.perf_counter() - t0) * 1000

        if stream:
            # Consume SSE data: lines
            full_text = ''
            for line in r.iter_lines():
                if line.startswith(b'data: '):
                    chunk = line[6:]
                    if chunk == b'[DONE]':
                        break
                    try:
                        obj = _json.loads(chunk)
                        full_text += obj.get('choices', [{}])[0].get('text', '')
                    except Exception:
                        pass
            tokens = len(full_text.split())
            text = full_text
        else:
            data = r.json()
            text = data.get('choices', [{}])[0].get('text', '')
            tokens = data.get('usage', {}).get('completion_tokens', len(text.split()))

        tok_per_sec = tokens / (elapsed_ms / 1000) if elapsed_ms > 0 else 0
        log_event('TURBO', f'{tokens} tokens @ {tok_per_sec:.1f} tok/s  level={level}  ctx_max={ctx_limit}t  backend=turboquant')
        return {'text': text, 'tokens': tokens, 'latency_ms': elapsed_ms,
                'tok_per_sec': tok_per_sec, 'backend': 'turboquant', 'level': level}
    except Exception as e:
        return {'text': None, 'tokens': 0, 'latency_ms': 0, 'error': str(e)}

# Test with RAG context
rag_ctx = retrieve(SAMPLE_QUERY, strategy='rag')
tq_result = try_turboquant(SAMPLE_QUERY, context=rag_ctx, level='turbo3')
if tq_result.get('text'):
    print(f'TurboQuant turbo3 ({tq_result["tok_per_sec"]:.1f} tok/s):\n{tq_result["text"]}')
else:
    print(f'⚠ TurboQuant not running: {tq_result.get("error")}')
    print('Setup: llama-server -m gemma4-legal-vlm-Q4_K_M.gguf -ctk turbo3 -ctv turbo3 --port 8090 --n-gpu-layers 35')

## §6 — 5-Tier Inference Router with Fallback Logic
**Like a decision tree for ordering food: first check if the fancy restaurant is open (E2B WebGPU), then the café (LiteRT), then fast food (ONNX), then delivery (KAG/RAG/DAG + server).**

Priority (highest → lowest):
1. **E2B WebGPU** — Transformers.js 4.x + WebGPU (not in Python — skip to LiteRT in Colab)
2. **LiteRT-LM** `:8070` — XNNPACK CPU or Intel iGPU OpenCL, MTP 4-head speedup
3. **ONNX 270M** — legacy Gemma 3 classification fallback
4. **TurboQuant** `:8090` — RTX 3060 Ti with turbo3 KV, full legal model
5. **Ollama** `:11434` — VLM support, fallback when TurboQuant not running

In [ ]:
OLLAMA_BASE_URL = 'http://127.0.0.1:11434'
_ollama_ready: bool | None = None
_ollama_check_expiry: float = 0.0

def is_ollama_ready() -> bool:
    global _ollama_ready, _ollama_check_expiry
    if _ollama_ready is not None and time.time() < _ollama_check_expiry:
        return _ollama_ready
    try:
        r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3)
        _ollama_ready = r.ok
    except Exception:
        _ollama_ready = False
    _ollama_check_expiry = time.time() + 60
    return _ollama_ready

def try_ollama(prompt: str, context: str = '', model: str = 'gemma4-legal:latest') -> dict:
    full_prompt = f'{context}\n\n{prompt}' if context else prompt
    payload = {'model': model, 'prompt': full_prompt, 'stream': False}
    t0 = time.perf_counter()
    try:
        r = requests.post(f'{OLLAMA_BASE_URL}/api/generate', json=payload, timeout=180)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        if not r.ok:
            return {'text': None, 'latency_ms': elapsed_ms, 'error': r.text}
        data = r.json()
        text = data.get('response', '')
        tokens = data.get('eval_count', len(text.split()))
        tok_per_sec = tokens / (elapsed_ms / 1000) if elapsed_ms > 0 else 0
        return {'text': text, 'tokens': tokens, 'latency_ms': elapsed_ms,
                'tok_per_sec': tok_per_sec, 'backend': 'ollama'}
    except Exception as e:
        return {'text': None, 'tokens': 0, 'latency_ms': 0, 'error': str(e)}

InferenceSource = Literal[
    'local-e2b', 'local-litert', 'local-onnx',
    'server-turboquant', 'server-ollama',
]

def route_inference(
    prompt: str,
    rag_strategy: Literal['rag', 'kag', 'dag'] = 'kag',
    compression: CompressionProfile | None = None,
) -> dict:
    """
    5-tier fallback router mirroring client-router.ts logic.
    Injects KAG/RAG/DAG context before any LLM call.
    """
    comp = compression or HW.compression

    # Assemble retrieval context first (always — runs in Python, no tier needed)
    ctx = retrieve(prompt, strategy=rag_strategy)
    log_event('ROUTER', f'Context assembled via {rag_strategy.upper()} ({len(ctx)} chars)')

    # Tier 1: E2B WebGPU — browser-only, skip in Python
    log_event('ROUTER', 'Tier 1 (E2B WebGPU): N/A in Python kernel — skipping')

    # Tier 2: LiteRT-LM (CPU/iGPU)
    if is_litert_ready():
        log_event('ROUTER', 'Tier 2 → LiteRT-LM :8070')
        result = litert_complete(prompt, context=ctx)
        if result.get('text'):
            result['source'] = 'local-litert'
            result['compression'] = comp.level.name
            return result

    # Tier 3: ONNX 270M (legacy — stub in Python)
    log_event('ROUTER', 'Tier 3 → ONNX 270M: browser-only, skip in Python')

    # Tier 4: TurboQuant llama-server (GPU)
    if is_turboquant_ready():
        log_event('ROUTER', f'Tier 4 → TurboQuant :8090 ({comp.kv_format})')
        result = try_turboquant(prompt, context=ctx, level=comp.kv_format if 'turbo' in comp.kv_format else 'turbo3')
        if result.get('text'):
            result['source'] = 'server-turboquant'
            result['compression'] = comp.level.name
            return result

    # Tier 5: Ollama (VLM-capable fallback)
    if is_ollama_ready():
        log_event('ROUTER', 'Tier 5 → Ollama :11434')
        result = try_ollama(prompt, context=ctx)
        if result.get('text'):
            result['source'] = 'server-ollama'
            result['compression'] = comp.level.name
            return result

    log_event('ROUTER', 'All tiers exhausted — returning empty result')
    return {'text': None, 'source': 'none', 'error': 'No inference backend available', 'context': ctx}

# Quick end-to-end test
final = route_inference(SAMPLE_QUERY, rag_strategy='kag')
print(f'Source  : {final.get("source")}')
print(f'Backend : {final.get("backend", "n/a")}')
print(f'Compress: {final.get("compression")}')
if final.get('tok_per_sec'):
    print(f'Speed   : {final["tok_per_sec"]:.1f} tok/s')
if final.get('text'):
    print(f'\nResponse:\n{final["text"][:400]}')
elif final.get('context'):
    print(f'\n(No LLM reached — retrieved context only):\n{final["context"][:400]}')

## §7 — Benchmark: turbo3 vs Q8_0 KV (CPU vs GPU)
**Like a race between two cars: the sports car (turbo3 on RTX 3060 Ti) vs the reliable sedan (Q8_0 on Intel 10th gen CPU) — who finishes first and how much fuel do they use?**

10 inference passes each. Measures latency (ms/token) and memory. Computes:
$$S = \frac{T_{Q8_0}}{T_{turbo3}}$$
where $T$ is ms/token (lower = better). $S > 1$ means turbo3 is faster.

In [ ]:
import matplotlib.pyplot as plt
import random

BENCH_PROMPTS = [
    'Is hearsay evidence admissible under federal rules?',
    'What constitutes a Brady violation in criminal procedure?',
    'Explain chain of custody requirements for DNA evidence.',
    'What is the exclusionary rule and when does it apply?',
    'Define stare decisis and its limits in federal courts.',
    'When can a warrant be issued without probable cause?',
    'What is the difference between direct and circumstantial evidence?',
    'Explain the fruit of the poisonous tree doctrine.',
    'What are the requirements for self-authentication under FRE 902?',
    'What is the burden of proof in a civil vs criminal case?',
]

N_PASSES = len(BENCH_PROMPTS)

def bench_backend(
    label: str,
    call_fn,  # callable(prompt, context) -> dict with latency_ms + tokens
    prompts: list[str],
) -> list[dict]:
    results = []
    for p in prompts:
        ctx = retrieve(p, strategy='rag')
        r = call_fn(p, ctx)
        if r.get('latency_ms') and r.get('tokens'):
            ms_per_tok = r['latency_ms'] / max(r['tokens'], 1)
            results.append({'prompt': p[:40], 'ms_per_tok': ms_per_tok, 'tokens': r['tokens'],
                            'latency_ms': r['latency_ms'], 'label': label})
        else:
            # Backend not available — simulate with realistic values for chart
            base = {'turbo3 (RTX 3060 Ti)': 8.0, 'Q8_0 (Intel 10th CPU)': 42.0}.get(label, 20.0)
            sim_ms_per_tok = base * random.uniform(0.85, 1.15)
            results.append({'prompt': p[:40], 'ms_per_tok': sim_ms_per_tok, 'tokens': 0,
                            'latency_ms': 0, 'label': label, 'simulated': True})
    return results

# Run benchmarks (real if backends up, simulated otherwise)
print('Benchmarking turbo3 (TurboQuant :8090) ...')
turbo3_results = bench_backend('turbo3 (RTX 3060 Ti)', lambda p, c: try_turboquant(p, c, 'turbo3'), BENCH_PROMPTS)

print('Benchmarking Q8_0 (LiteRT CPU :8070) ...')
q8_results = bench_backend('Q8_0 (Intel 10th CPU)', lambda p, c: litert_complete(p, c), BENCH_PROMPTS)

# Summary stats
import statistics
t3_vals = [r['ms_per_tok'] for r in turbo3_results]
q8_vals = [r['ms_per_tok'] for r in q8_results]
t3_mean = statistics.mean(t3_vals)
q8_mean = statistics.mean(q8_vals)
speedup = q8_mean / t3_mean

print(f'\n{"Backend":<26} {"Mean ms/tok":>12} {"Median ms/tok":>14}')
print('-' * 55)
print(f'{"turbo3 (RTX 3060 Ti)":<26} {t3_mean:>12.2f} {statistics.median(t3_vals):>14.2f}')
print(f'{"Q8_0 (Intel 10th CPU)":<26} {q8_mean:>12.2f} {statistics.median(q8_vals):>14.2f}')
print(f'\nSpeedup S = T_Q8 / T_turbo3 = {speedup:.2f}×  (>1 means turbo3 faster)')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
labels = [r['prompt'] for r in turbo3_results]
x = range(len(labels))
axes[0].bar([i - 0.2 for i in x], t3_vals, 0.4, label='turbo3 (GPU)', color='#4CAF50')
axes[0].bar([i + 0.2 for i in x], q8_vals, 0.4, label='Q8_0 (CPU)', color='#2196F3')
axes[0].set_xticks(list(x))
axes[0].set_xticklabels(labels, rotation=45, ha='right', fontsize=7)
axes[0].set_ylabel('ms / token (lower = faster)')
axes[0].set_title('turbo3 vs Q8_0 KV — latency per token')
axes[0].legend()
axes[0].axhline(t3_mean, color='#4CAF50', linestyle='--', alpha=0.5, label=f'turbo3 mean {t3_mean:.1f}')
axes[0].axhline(q8_mean, color='#2196F3', linestyle='--', alpha=0.5, label=f'Q8_0 mean {q8_mean:.1f}')

axes[1].bar(['turbo3\n(RTX 3060 Ti)', 'Q8_0\n(Intel 10th CPU)'], [t3_mean, q8_mean],
            color=['#4CAF50', '#2196F3'])
axes[1].set_ylabel('Mean ms / token')
axes[1].set_title(f'Average latency — Speedup = {speedup:.2f}×')
for i, v in enumerate([t3_mean, q8_mean]):
    axes[1].text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold')

any_sim = any(r.get('simulated') for r in turbo3_results + q8_results)
if any_sim:
    fig.text(0.5, 0.01, '⚠ Simulated values — backends not running. Run Track 1 + Track 2 for real results.',
             ha='center', color='orange', fontsize=9)

plt.tight_layout()
plt.savefig('benchmark_turbo3_vs_q8.png', dpi=120, bbox_inches='tight')
plt.show()
log_event('BENCH', f'Benchmark complete. turbo3={t3_mean:.1f}ms/tok  Q8_0={q8_mean:.1f}ms/tok  speedup={speedup:.2f}×')

## §8 — GRPO Adapter Merge + GGUF Export (Track 3)
**Like permanently sewing a patch onto a jacket instead of just pinning it — then folding it into a compact package (GGUF) that any llama.cpp-based server can unpack.**

1. Load Gemma 4 E4B base weights from HuggingFace
2. Attach GRPO LoRA adapter (`Semaj90/gemma4-e4b-legal-grpo`)
3. Call `merge_and_unload()` — fuses adapter weights into base, drops LoRA overhead
4. Export to GGUF via llama.cpp `convert_hf_to_gguf.py` script
5. Validate: check file size (~5 GB) and SHA256 checksum

**Run this section on Colab A100 (Track 3) — requires 16 GB+ VRAM.**

In [ ]:
import os
import hashlib
import shutil
from pathlib import Path

# ── Configuration (set before running) ───────────────────────────────────
BASE_MODEL_ID = 'google/gemma-4-e4b-it'
ADAPTER_ID    = 'Semaj90/gemma4-e4b-legal-grpo'
MERGED_DIR    = Path('./gemma4-legal-vlm-merged')
GGUF_DIR      = Path('./gemma4-legal-vlm-gguf')
GGUF_QUANT    = 'Q4_K_M'   # target quantization

# ── Guard: skip gracefully if not on Colab A100 ──────────────────────────
try:
    import torch
    _has_gpu = torch.cuda.is_available()
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if _has_gpu else 0
except Exception:
    _has_gpu, _vram = False, 0.0

if not _has_gpu or _vram < 16:
    print(f'⚠ Skipping merge+export — requires 16 GB+ VRAM (detected {_vram:.1f} GB).')
    print('Run this section on Colab A100 (Track 3).')
    log_event('MERGE', f'Skipped — VRAM={_vram:.1f}GB (need 16GB+)')
else:
    # Step 1: Load base model + attach adapter
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    log_event('MERGE', f'Loading base model {BASE_MODEL_ID}')
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

    log_event('MERGE', f'Attaching GRPO adapter {ADAPTER_ID}')
    model = PeftModel.from_pretrained(base_model, ADAPTER_ID)

    # Step 2: Merge adapter into base weights
    log_event('MERGE', 'Merging adapter (merge_and_unload)...')
    model = model.merge_and_unload()

    # Step 3: Save merged model
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    log_event('MERGE', f'Saved merged model to {MERGED_DIR}')

    # Step 4: Convert to GGUF using llama.cpp
    llamacpp_script = Path('./llama.cpp/convert_hf_to_gguf.py')
    if not llamacpp_script.exists():
        log_event('GGUF', 'Cloning llama.cpp for GGUF conversion...')
        subprocess.run(['git', 'clone', 'https://github.com/ggerganov/llama.cpp.git', '--depth=1'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'llama.cpp/requirements.txt'], check=True)

    GGUF_DIR.mkdir(parents=True, exist_ok=True)
    gguf_path = GGUF_DIR / f'gemma4-legal-vlm-{GGUF_QUANT}.gguf'
    log_event('GGUF', f'Converting to GGUF ({GGUF_QUANT})...')
    subprocess.run([
        sys.executable, str(llamacpp_script),
        str(MERGED_DIR),
        '--outfile', str(gguf_path),
        '--outtype', GGUF_QUANT.lower().replace('_', ''),
    ], check=True)

    # Step 5: Validate
    size_gb = gguf_path.stat().st_size / 1e9
    sha256 = hashlib.sha256(gguf_path.read_bytes()).hexdigest()
    log_event('GGUF', f'GGUF exported: {gguf_path.name}  size={size_gb:.2f}GB  sha256={sha256[:16]}...')
    print(f'✓ GGUF file: {gguf_path}')
    print(f'  Size    : {size_gb:.2f} GB  (expected ~4.5-5.5 GB for E4B Q4_K_M)')
    print(f'  SHA256  : {sha256}')

    # Next step: Deploy via Ollama
    print(f'''
Next steps (Track 1 + Track 2):
  ollama create gemma4-legal-vlm:latest -f Modelfile   # Modelfile points to GGUF
  llama-server -m {gguf_path} -ctk turbo3 -ctv turbo3 --port 8090 --n-gpu-layers 35
''')